# Qwen3.5 VL Image-to-FOL

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import os
import subprocess
import tempfile
import json
from pathlib import Path
from outlines.types import CFG
from outlines.inputs import Chat
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import torch
import outlines

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-9B"
MAX_NEW_TOKENS = 1024

# https://huggingface.co/Qwen/Qwen3.5-0.8B#best-practices
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.2

BASE_DIR = Path.cwd().parent
CATEGORY = "train"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

STATE_FILE = BASE_DIR / "smt_state.json"
SMT_FILE = BASE_DIR / "smt.json"

CVC5_PATH = Path.home() / "cvc5-Linux-x86_64-shared" / "bin" / "cvc5"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use torch.float16 if your GPU is older and doesn't support bfloat16
    bnb_4bit_use_double_quant=True,  # Optional: Saves a bit more memory
    bnb_4bit_quant_type="nf4",  # Optional: Recommended for better performance
)

model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto", quantization_config=bnb_config
    ),
    AutoTokenizer.from_pretrained(MODEL_ID),
)

In [ ]:
# A stripped down EBNF conforming to strict guided decoding rules.
# 1. Comments are mathematically impossible.
# 2. Infinite loops are prevented by a forced exit lifecycle.
# 3. Type-safety is enforced lexically via _entity and s_ prefixes.
# 4. Hardcoding the preamble signatures
# 5. Prohibit the LLM from re-declaring preamble functions or using generic sorts for Entities/Series
# 6. Use a Phase-Locked Grammar. This forces the LLM to follow a one-way path: first declarations, then data anchors, then logic, then exit.
# 7. We limit underscores and character length to prevent 'series_series_series' loops
# 8. Prevent an infinite token loop by removing the * quantifier and physically cap the number of allowed repetitions.

# Shared terminals for both passes to ensure regex matching is identical
SHARED_TERMINALS = r"""
?term: spec_constant | ENTITY_SYM | SERIES_SYM | LOGIC_VAR_BOOL | LOGIC_VAR_REAL | PREAMBLE_CONST | preamble_call | "(" RESERVED_OP term+ ")"

spec_constant: DECIMAL | NUMERAL | STRING_LIT

preamble_call: "(" "f" SERIES_SYM term ")"
             | "(" "attr" SERIES_SYM ENTITY_SYM ")"
             | "(" "name_of" ENTITY_SYM ")"
             | "(" "diff" term term ")"
             | "(" "is_gt" SERIES_SYM SERIES_SYM term ")"
             | "(" "is_eq" SERIES_SYM SERIES_SYM term ")"
             | "(" "is_inc" SERIES_SYM term term ")"
             | "(" "is_dec" SERIES_SYM term term ")"
             | "(" "is_at_val" SERIES_SYM term term ")"

PREAMBLE_CONST: "epsilon" | "AnsBool" | "AnsReal" | "AnsString"
RESERVED_OP: "=" | "not" | "and" | "or" | ">" | "<" | ">=" | "<=" | "ite"

# FIXED REGEXES: Added A-Z, removed strict {} counts, added -? for negatives, relaxed string contents
ENTITY_SYM: /[a-zA-Z0-9_]+_entity/
SERIES_SYM: /s_[a-zA-Z0-9_]+/
LOGIC_VAR_BOOL: /[a-zA-Z0-9_]+_(drop|inc|dec|const|stable|bool)/
LOGIC_VAR_REAL: /[a-zA-Z0-9_]+_(val|max|min|coord|cycle)/

NUMERAL: /-?[0-9]+/
DECIMAL: /-?[0-9]+\.[0-9]+/
STRING_LIT: /"[^"]*"/

%import common.WS
%ignore WS
"""

# PASS 1: All Declarations (Entities, Series, Reals, and Booleans)
# Focus: Extraction, Transcription, and Schema Declaration

SMT_LIB_GRAMMAR_PASS1 = (
    r"""
?start: script

script: world_entities data_anchors result_vars

# 1. Grounding: Who and what are in the image?
world_entities: (entity_decl | series_decl | attr_assert | name_assert)+

# 2. Transcription: What are the raw data points?
# Note: Capped at 30 anchors to prevent infinite point extraction loops
data_anchors: anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item? anchor_item?
anchor_item: real_decl | anchor_assert

# 3. Schema: CAPPED to prevent infinite hallucination loops!
# Small models will loop infinitely if we use `*`. We force a maximum of 8 variables.
result_vars: var_decl? var_decl? var_decl? var_decl? var_decl? var_decl? var_decl? var_decl?
var_decl: bool_decl | real_decl

entity_decl: "(" "declare-const" ENTITY_SYM "Entity" ")"
series_decl: "(" "declare-const" SERIES_SYM "Series" ")"
real_decl:   "(" "declare-const" LOGIC_VAR_REAL "Real" ")"
bool_decl:   "(" "declare-const" LOGIC_VAR_BOOL "Bool" ")"

name_assert:   "(" "assert" "(" "=" "(" "name_of" ENTITY_SYM ")" STRING_LIT ")" ")"
attr_assert:   "(" "assert" "(" "attr" SERIES_SYM ENTITY_SYM ")" ")"
anchor_assert: "(" "assert" "(" "=" "(" "f" SERIES_SYM term ")" term ")" ")"
"""
    + SHARED_TERMINALS
)

# PASS 2: Pure Logic Assertions & Commands
SMT_LIB_GRAMMAR_PASS2 = (
    r"""
?start: script

script: logic_assert+ check_sat_cmd get_value_cmd exit_cmd

# Logic assertions use ONLY the variables declared in Pass 1
logic_assert:  "(" "assert" "(" "=" (LOGIC_VAR_BOOL | LOGIC_VAR_REAL) term ")" ")"
             | "(" "assert" "(" RESERVED_OP term+ ")" ")"

check_sat_cmd: "(" "check-sat" ")"
get_value_cmd: "(" "get-value" "(" (LOGIC_VAR_BOOL | LOGIC_VAR_REAL | PREAMBLE_CONST | ENTITY_SYM | SERIES_SYM)+ ")" ")"
exit_cmd: "(" "exit" ")"
"""
    + SHARED_TERMINALS
)

SMT_CFG_PASS1 = CFG(SMT_LIB_GRAMMAR_PASS1)
SMT_CFG_PASS2 = CFG(SMT_LIB_GRAMMAR_PASS2)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen3.5 (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
PROMPT_YES_NO_PASS1 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Yes/No
Question: {question}

[TASK]
You are a formal logic extractor (PASS 1/2: Extraction). Your goal is to build a "Knowledge Base" from the visual evidence.
1. Declare all visual Entities and Series.
2. Assert their names, attributes, and data anchors (coordinates).
3. Pre-declare the logical variables (Bool/Real) you will need to perform reasoning in Pass 2.

[STRICT RULES]
- NO REASONING: Do not perform comparisons or logic yet. Just record facts.
- SCHEMA LOCK: You must declare any variable (e.g., max_val_real) here, or you won't be able to use it in Pass 2.
- ANCHORS: Numeric values must be exact extractions from the visible axes.

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[EXAMPLE]
(declare-const s2p_entity Entity)
(assert (= (name_of s2p_entity) "S2p"))
(declare-const s_trace Series)
(assert (attr s_trace s2p_entity))
;; Visual Data Anchors
(assert (= (f s_trace 0.0) 0.0))
(assert (= (f s_trace 10.0) 1.8))
(assert (= (f s_trace 20.0) 1.6))
(assert (= (f s_trace 30.0) 1.6))
(assert (= (f s_trace 50.0) 1.9))
(assert (= (f s_trace 75.0) 1.2))
;; Schema Reservations
(declare-const max_val_real Real)
(declare-const ans_bool Bool)
"""

PROMPT_YES_NO_PASS2 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Yes/No
Question: {question}

[TASK]
You are a formal logic reasoning agent (PASS 2/2: Query). Use the pre-declared Knowledge Base to formulate the logical proof for the question.

[STRICT RULES]
- NO DECLARATIONS: Do not use 'declare-const'. Use only the variables provided above.
- FINITE SEARCH: Use (or ...) for maxima/minima over the extracted x-values.
- OUTPUT: End with (check-sat), (get-value), and (exit).

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[KNOWLEDGE BASE (FROM PASS 1)]
{declarations}

[EXAMPLE]
(assert (or
  (= max_val_real (f s_trace 0.0))
  (= max_val_real (f s_trace 10.0))
  (= max_val_real (f s_trace 20.0))
  (= max_val_real (f s_trace 30.0))
  (= max_val_real (f s_trace 50.0))
  (= max_val_real (f s_trace 75.0))
))
(assert (and
  (>= max_val_real (f s_trace 0.0))
  (>= max_val_real (f s_trace 10.0))
  (>= max_val_real (f s_trace 20.0))
  (>= max_val_real (f s_trace 30.0))
  (>= max_val_real (f s_trace 50.0))
  (>= max_val_real (f s_trace 75.0))
))
(assert (= ans_bool (> max_val_real 2.0)))
(check-sat)
(get-value (ans_bool))
(exit)
"""

In [ ]:
PROMPT_FACTOID_PASS1 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Factoid
Question: {question}

[TASK]
(PASS 1/2: EXTRACTION) Build a formal Knowledge Base by extracting explicitly visible visual entities and naming them. You must also declare any Boolean or Real variables needed for the final reasoning logic.

[STRICT RULES]
- NO LOGIC: Do not perform comparisons or conditional logic in this pass.
- DECLARE ALL: Every Entity and reasoning variable (Bool/Real) must be declared here.
- NO QUANTIFIERS: Do NOT use 'forall' or 'exists'.

[AVAILABLE SMT-LIB ENVIRONMENT (ALREADY LOADED)]
{preamble}

[EXAMPLE]
(declare-const structural_region_entity Entity)
(assert (= (name_of structural_region_entity) "Multi-quantum well"))
(declare-const sidewall_entity Entity)
(assert (= (name_of sidewall_entity) "sidewall"))
(declare-const identified_issue_entity Entity)
(assert (= (name_of identified_issue_entity) "Etch damage layer"))
;; Pre-declaring the logical relationship flags
(declare-const sidewall_is_part_of_region_bool Bool)
(declare-const issue_forms_on_sidewall_bool Bool)
"""

PROMPT_FACTOID_PASS2 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Factoid
Question: {question}

[TASK]
(PASS 2/2: REASONING) Formulate the logical assertions and solver commands to answer the question.
Use ONLY the variables declared in the Knowledge Base above.

[STRICT RULES]
- NO DECLARATIONS: You are strictly forbidden from using `declare-const`.
- DIRECT ASSERTIONS: Use the pre-declared Boolean/Real variables to represent your logic.
- SOLVER COMMANDS: Always end with (check-sat), (get-value), and (exit).

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[KNOWLEDGE BASE (FROM PASS 1)]
{declarations}

[EXAMPLE]
(assert (= sidewall_is_part_of_region_bool true))
(assert (= issue_forms_on_sidewall_bool true))
(assert (ite (and sidewall_is_part_of_region_bool issue_forms_on_sidewall_bool)
             (= AnsString (name_of identified_issue_entity))
             (= AnsString "Unknown")))
(check-sat)
(get-value (AnsString))
(exit)
"""

In [ ]:
PROMPT_LIST_PASS1 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: List
Question: {question}

[TASK]
You are a formal logic extractor (Phase 1: Knowledge Base).
1. Identify all visual entities and series.
2. Extract discrete numeric anchors from the visual evidence.
3. Pre-declare the result variables (e.g., rank1_entity, rank2_entity) needed for the final list.

[STRICT RULES]
- NO QUANTIFIERS: Do NOT use 'forall' or 'exists'.
- DISCRETE ANCHORS: Numeric extraction must rely on exact coordinates.
- NO LOGIC: Do not perform comparisons or sorting in this phase. Just record the facts.

[AVAILABLE SMT-LIB ENVIRONMENT (ALREADY LOADED)]
{preamble}

[EXAMPLE]
(declare-const e_CF2_entity Entity) (assert (= (name_of e_CF2_entity) "CF2"))
(declare-const e_CF_entity Entity) (assert (= (name_of e_CF_entity) "CF"))
(declare-const e_C_entity Entity) (assert (= (name_of e_C_entity) "C"))
(declare-const s_CF2 Series) (assert (attr s_CF2 e_CF2_entity))
(declare-const s_CF Series) (assert (attr s_CF e_CF_entity))
(declare-const s_C Series) (assert (attr s_C e_C_entity))
(assert (= (f s_CF2 6.25) 5.75))
(assert (= (f s_CF 6.25) 4.15))
(assert (= (f s_C 6.25) 2.50))
;; Pre-declaring the slots for the final answer
(declare-const rank1_entity Entity)
(declare-const rank2_entity Entity)
(declare-const rank3_entity Entity)
"""

PROMPT_LIST_PASS2 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: List
Question: {question}

[TASK]
You are a formal logic reasoning agent (Phase 2: Query).
Using ONLY the data anchors and entities provided above, formulate the comparison logic to determine the requested order/list.

[STRICT RULES]
- NO NEW DECLARATIONS: You are forbidden from using 'declare-const'. Use only what was provided in Pass 1.
- EXPLICIT RANKING: Use assertions to map specific entities to the rank variables.

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[KNOWLEDGE BASE (FROM PASS 1)]
{declarations}

[EXAMPLE]
(assert (and
    (is_gt s_CF2 s_CF 6.25)
    (is_gt s_CF s_C 6.25)
))
(assert (and
    (= rank1_entity e_CF2_entity)
    (= rank2_entity e_CF_entity)
    (= rank3_entity e_C_entity)
))
(check-sat)
(get-value ((name_of rank1_entity) (name_of rank2_entity) (name_of rank3_entity)))
(exit)
"""

In [ ]:
PROMPT_PARAGRAPH_PASS1 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Paragraph
Question: {question}

[TASK]
You are a formal logic extractor (Phase 1: Knowledge Base).
1. Identify all relevant visual entities and series.
2. Extract discrete numeric anchors (x, y coordinates) from the image.
3. Pre-declare the Boolean or Real variables you will need to answer the question logic.

[STRICT RULES]
- NO LOGIC: Do not use 'is_inc', 'is_dec', or comparisons here. Just transcribe facts.
- PRE-DECLARATION: Every variable used in the next phase (e.g., trend_bool) must be declared now.

[EXAMPLE]
(declare-const o1s_entity Entity)
(assert (= (name_of o1s_entity) "O1s"))
(declare-const s_o1s Series)
(assert (attr s_o1s o1s_entity))
(assert (= (f s_o1s 0.0) 16.0))
(assert (= (f s_o1s 10.0) 10.5))
(assert (= (f s_o1s 75.0) 7.5))
(declare-const o_initial_drop_bool Bool)
(declare-const o_steady_decrease_bool Bool)
"""

PROMPT_PARAGRAPH_PASS2 = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Paragraph
Question: {question}

[TASK]
You are a formal logic reasoning agent (Phase 2: Query).
Using ONLY the data anchors and variables provided in the Knowledge Base above, formulate the logical assertions to answer the question.

[STRICT RULES]
- NO DECLARATIONS: Do NOT use 'declare-const'.
- USE PREAMBLE: Use functions like 'is_dec', 'is_inc', and 'is_eq' to assign values to the pre-declared Booleans.

[AVAILABLE SMT-LIB ENVIRONMENT]
{preamble}

[KNOWLEDGE BASE (FROM PASS 1)]
{declarations}

[EXAMPLE]
(assert (= o_initial_drop_bool (is_dec s_o1s 0.0 10.0)))
(assert (= o_steady_decrease_bool (is_dec s_o1s 10.0 75.0)))
(check-sat)
(get-value (o_initial_drop_bool o_steady_decrease_bool))
(exit)
"""

In [ ]:
PROMPTS = {
    "Yes/No": {1: PROMPT_YES_NO_PASS1, 2: PROMPT_YES_NO_PASS2},
    "Factoid": {1: PROMPT_FACTOID_PASS1, 2: PROMPT_FACTOID_PASS2},
    "List": {1: PROMPT_LIST_PASS1, 2: PROMPT_LIST_PASS2},
    "Paragraph": {1: PROMPT_PARAGRAPH_PASS1, 2: PROMPT_PARAGRAPH_PASS2},
}

In [ ]:
PREAMBLE = """
;; --- UNIVERSAL PREAMBLE ---
;; Logic: Combined Linear Real Arithmetic and Strings (supported by Z3/CVC5)
(set-logic ALL)

;; 1. TYPE DEFINITIONS
(declare-sort Series 0)
(declare-sort Entity 0)
(declare-const epsilon Real)
(assert (= epsilon 0.02))

;; 2. CORE MAPPING FUNCTIONS
(declare-fun f (Series Real) Real)
(declare-fun attr (Series Entity) Bool)
(declare-fun name_of (Entity) String)
(declare-fun unit_of (Series) String)

;; 3. GEOMETRIC & TREND PRIMITIVES
(define-fun diff ((a Real) (b Real)) Real
  (ite (>= (- a b) 0.0) (- a b) (- b a)))

;; Comparison & Trend
(define-fun is_gt ((s1 Series) (s2 Series) (x Real)) Bool
  (> (- (f s1 x) (f s2 x)) epsilon))

(define-fun is_eq ((s1 Series) (s2 Series) (x Real)) Bool
  (<= (diff (f s1 x) (f s2 x)) epsilon))

(define-fun is_inc ((s Series) (x1 Real) (x2 Real)) Bool
  (> (- (f s x2) (f s x1)) epsilon))

(define-fun is_dec ((s Series) (x1 Real) (x2 Real)) Bool
  (> (- (f s x1) (f s x2)) epsilon))

;; 4. EXTREMA & INTERSECTION
(define-fun is_at_val ((s Series) (x Real) (y Real)) Bool
  (<= (diff (f s x) y) epsilon))

(define-fun is_peak ((s Series) (x_l Real) (x_m Real) (x_r Real)) Bool
  (and (>= (f s x_m) (f s x_l)) (>= (f s x_m) (f s x_r))))

;; 5. STANDARDIZED OUTPUT VARIABLES
(declare-const AnsBool Bool)
(declare-const AnsReal Real)
(declare-const AnsString String)
"""

In [ ]:
def get_paper_context(json_file_path, window_size=2):
    """
    Finds the parent content.json, extracts the image caption, and
    grabs a sliding window of text blocks (e.g., 2 before, 2 after)
    surrounding the image for highly targeted context.
    """
    # Navigate up from .../16/images/fig_2.json to .../16/content.json
    content_json_path = json_file_path.parent.parent / "content.json"

    assert content_json_path.exists(), f"{content_json_path}"

    # The image path as it appears in content.json (e.g., "images/fig_2.jpg")
    target_img_path = f"images/{json_file_path.stem}.jpg"

    with open(content_json_path, "r", encoding="utf-8") as f:
        content_data = json.load(f)

    img_index = -1
    caption_text = ""

    # Locate the image block in the flat JSON array
    for idx, block in enumerate(content_data):
        if block.get("type") == "image" and block.get("img_path") == target_img_path:
            img_index = idx
            if "img_caption" in block and block["img_caption"]:
                caption_text = " ".join(block["img_caption"])
            break

    if img_index == -1:
        return "Specific context not found for this image."

    # Gather text blocks BEFORE the image
    text_before = []
    for i in range(img_index - 1, -1, -1):
        block = content_data[i]
        if block.get("type") == "text" and "text" in block:
            text_before.insert(0, block["text"])  # Keep chronological order
            if len(text_before) == window_size:
                break

    # Gather text blocks AFTER the image
    text_after = []
    for i in range(img_index + 1, len(content_data)):
        block = content_data[i]
        if block.get("type") == "text" and "text" in block:
            text_after.append(block["text"])
            if len(text_after) == window_size:
                break

    # Assemble the final context string
    context_blocks = []
    if caption_text:
        context_blocks.append(f"Image Caption: {caption_text}")

    context_blocks.extend(text_before)
    context_blocks.extend(text_after)

    return "\n\n".join(context_blocks)

In [ ]:
def validate_smt(code: str) -> tuple[bool, str]:
    """
    Validates SMT-LIB code by executing cvc5.
    Returns: (bool: is_satisfiable, str: output_message)
    """
    # 1. Write to a secure temporary file
    with tempfile.NamedTemporaryFile(mode="w", suffix=".smt2", delete=False) as tf:
        tf.write(code)
        temp_path = tf.name

    try:
        # 2. Execute solver with strict timeout and specific flags
        # --produce-models is necessary if you plan to call (get-model) later
        result = subprocess.run(
            [
                CVC5_PATH,
                "--lang",
                "smt2",
                "--produce-models",
                "--incremental",
                temp_path,
            ],
            capture_output=True,
            text=True,
            timeout=10,
        )

        stdout = result.stdout.strip()
        stderr = result.stderr.strip()

        # 3. Precise Status Parsing
        # The first non-empty line of SMT-LIB output is typically the status
        lines = [line for line in stdout.split("\n") if line.strip()]
        status = lines[0].lower() if lines else ""

        if stderr or "error" in stdout.lower():
            return False, stderr if stderr else stdout
        if "sat" == status:
            return True, stdout
        elif "unsat" == status:
            return False, stdout
        elif "unknown" == status:
            return False, stdout
        else:
            return False, f"Unexpected Solver Output: {stdout}"
    except subprocess.TimeoutExpired:
        return (
            False,
            "Timeout: The solver took too long (potential infinite search space).",
        )
    except Exception as e:
        return False, f"Internal Execution Error: {str(e)}"
    finally:
        # Ensure cleanup even if execution fails
        if os.path.exists(temp_path):
            os.remove(temp_path)


def reflect(image, q_obj, paper_context, max_retries=3, verbose=False, **gen_kwargs):
    question_text = q_obj.get("question") or q_obj.get("questions")
    question_type = q_obj.get("question_type", "")
    answer_type = q_obj.get("answer_type", "")

    # --- INITIAL PROMPT SETUP ---
    first_pass_text = PROMPTS[answer_type][1].format(
        question=question_text,
        question_type=question_type,
        answer_type=answer_type,
        context=paper_context,
        preamble=PREAMBLE,
    )

    prompt_pass1 = Chat(
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": first_pass_text},
                ],
            }
        ]
    )

    # ==========================================
    # PHASE 1: Generate and Validate Declarations
    # ==========================================
    declarations = ""
    pass1_success = False

    for attempt in range(max_retries):
        declarations = model(prompt_pass1, SMT_CFG_PASS1, **gen_kwargs)

        # Create a temporary SMT script just to check if the declarations are valid
        test_smt_pass1 = f"""
        ;; --- [PREAMBLE] ---
        {PREAMBLE}
        ;; --- [PASS 1: Data & Declarations] Attempt {attempt + 1} ---
        {declarations}
        (check-sat)
        """

        pass1_success, output = validate_smt(test_smt_pass1)

        if verbose:
            print(
                f"[PASS 1 - Attempt {attempt + 1}]\n[Code]\n{test_smt_pass1}\n[Output]\n{output}\n"
            )

        if pass1_success:
            break

        # If it fails, it's usually a duplicate declaration or syntax error. Reflect and retry.
        reflection_text = (
            f"The SMT solver rejected your declarations with this error:\n{output}\n\n"
            f"Please correct the extraction. Ensure you are not declaring the same variable twice."
        )

        prompt_pass1.add_assistant_message([{"type": "text", "text": declarations}])
        prompt_pass1.add_user_message([{"type": "text", "text": reflection_text}])

    if not pass1_success:
        return None, "Failed to generate valid Pass 1 declarations after retries."

    # ==========================================
    # PHASE 2: Generate and Validate Logic
    # ==========================================
    second_pass_text = PROMPTS[answer_type][2].format(
        question=question_text,
        question_type=question_type,
        answer_type=answer_type,
        context=paper_context,
        preamble=PREAMBLE,
        declarations=declarations,  # Safely use our verified declarations
    )

    prompt_pass2 = Chat(
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": second_pass_text},
                ],
            }
        ]
    )

    for attempt in range(max_retries):
        logic = model(prompt_pass2, SMT_CFG_PASS2, **gen_kwargs)

        # Combine everything for the final solver check
        final_smt = f"""
        ;; --- [PREAMBLE] ---
        {PREAMBLE}
        ;; --- [PASS 1: Data & Declarations] ---
        {declarations}
        ;; --- [PASS 2: Logic & Execution] Attempt {attempt + 1} ---
        {logic}
        """

        success, output = validate_smt(final_smt)

        if verbose:
            print(
                f"[PASS 2 - Attempt {attempt + 1}]\n[Code]\n{final_smt}\n[Output]\n{output}\n"
            )

        if success:
            return final_smt, output

        # If Pass 2 fails, reflect and retry Pass 2
        reflection_text = (
            f"The SMT solver failed with this output:\n{output}\n\n"
            f"Fix the logic. You MAY NOT declare new entities or series. Use only the provided variables."
        )
        prompt_pass2.add_assistant_message([{"type": "text", "text": logic}])
        prompt_pass2.add_user_message([{"type": "text", "text": reflection_text}])

    return None, "Failed to reach a valid logical formulation in Pass 2 after retries."

In [ ]:
json_files = [
    f
    for f in CASE_DIR.rglob("*.json")
    if "content.json" not in f.name and "images" in str(f) and ".vscode" not in str(f)
]

In [ ]:
ANSWER_TYPE = "Paragraph"

filename = None
q_index = None
for file_path in json_files:
    with open(file_path, "r") as f:
        data = json.load(f)

    sub_key = list(data["bbox"].keys())[0]

    if "vqa" not in data:
        continue

    for i, q_obj in enumerate(data["vqa"][sub_key]):
        if q_obj.get("answer_type", "") == ANSWER_TYPE:
            filename = file_path
            q_index = i
            break

    if filename is not None:
        break

In [ ]:
with open(filename, "r") as f:
    data = json.load(f)

img = Image.open(filename.with_suffix(".jpg"))
box = data["bbox"][sub_key]

crop = img.crop((box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]))
print(f"Previewing Subfigure: {sub_key}")
display(crop)

In [ ]:
reflect(
    crop,
    data["vqa"][sub_key][q_index],
    get_paper_context(filename),
    max_retries=3,
    verbose=True,
    do_sample=True,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    min_p=MIN_P,
    top_p=TOP_P,
    top_k=TOP_K,
    # presence_penalty=PRESENCE_PENALTY, # NOT USED!
    repetition_penalty=REPETITION_PENALTY,
)